# Interactive Script: **Co-register Data Cube**

**Author:** Baturalp Arisoy<br>
**Contact:** baturalp.arisoy@uni-wuerzburg.de - Call me Batu :)

## Overview
Sentinel-2 scenes are geolocated to within roughly half a pixel, and that residual shift moves from scene to scene. In a time series it shows up as the whole image jittering between dates, which blurs any per-pixel analysis: NDVI trends, change detection, anything read at one coordinate through time.

Co-registration measures the shift of each scene against a reference scene and resamples the cube once so every date lands on the same ground.

## Contents

1. [Before You Start](#1-before-you-start)
2. [Co-register a Cube](#2-co-register-a-cube)
3. [Cubes That Keep Their Clouds](#3-cubes-that-keep-their-clouds)
4. [Check the Result](#4-check-the-result)
5. [Spectral Profile Before and After](#5-spectral-profile-before-and-after)

---

## 1. Before You Start

Three things decide whether this works at all.

**Area size and surface.** Matching needs texture. A large, heterogeneous area (fields, roads, coastline, settlement) gives many usable windows; a small or homogeneous patch (dense forest, snow, open water) gives few or none. For a small area of interest, build the cube over a larger box, co-register, and clip afterwards with `clip_stac` (notebook 1, chapter 8).

**Clouds.** A cloudy scene cannot be matched. The algorithm places its matching windows only where both the scene and its reference are clear, so a cloud-masked cube performs much better than an unmasked one. Either build with `cloud_masking=True`, or mask with s2cloudless first (notebook 2), or pass a binary mask with `cloud_mask=` (chapter 3 below).

**Series length.** On a handful of dates a few days apart the scenes are usually already aligned, and there is nothing to see. Build a season or a year to judge the effect.

**Shape of the cube.** Build with `clip_raster=False`. A rectangle gives the matcher the most room; a cut-out polygon leaves no-data around the edges that the windows have to avoid.

In [ ]:
from stac2cube import coregister_cube, show_coregistration_parameter_help

In [ ]:
show_coregistration_parameter_help()

## 2. Co-register a Cube

The engine estimates the shift of each scene at up to `grid_size` x `grid_size` + 1 window positions and combines them into a consensus, so an outlier window over a cloud or a moving surface is outvoted rather than believed. Scenes chain scene to scene, and the pixels are resampled exactly **once** at the end regardless of how many estimation passes ran.

Leave `output_path=None` to write next to the input with a `_cr` suffix.

In [ ]:
out = coregister_cube(
    input_path="../results/test.nc",
    grid_size=7,                      # window budget of the scan
    match_band="auto",                # first available native 10-m band: nir, red, green, blue
    first_scene_mode="auto",          # pick the most textured low-cloud scene as anchor
    max_cc=100,                       # exclude scenes cloudier than this (needs a masked cube)
    time_period=None,                 # or ["YYYY-MM-DD", "YYYY-MM-DD"]
    min_inliers_keep="auto",          # windows that must agree for a scene to be kept
    min_inliers_update_ref="auto",    # windows that must agree for a scene to become reference
    max_cloud_update_ref=20.0,        # cloudier scenes never become the reference
    iteration=1,                      # "auto" measures the cube and stops when it stops improving
    adaptive=True,                    # coarse scan first, full budget only where it is unclear
    output_path=None,                 # None writes <input>_cr.nc
)

### 2.1 Reading the summary

The run prints a per-scene table and a summary. Scenes that could not be matched are **dropped** from the cube, which is why the co-registered cube can be shorter than the input. That doubles as an automatic filter: a scene nothing can be matched against is usually unusable anyway.

If the summary drops scenes you know are clear and usable:

- raise `grid_size`, which adds voters and usually solves it;
- the area may be too small, see chapter 1;
- the surface may be too homogeneous, so try a period with more contrast, for example the first half of the growing season;
- try `first_scene_mode="auto"` if you anchored at a fixed scene, or name a date you have looked at and know is clean.

### 2.2 Choosing the anchor

The whole chain hangs off the first reference, so a hazy or snow-covered anchor degrades everything downstream.

| `first_scene_mode` | behaviour |
|---|---|
| `"auto"` | picks the most textured low-cloud scene of the series, then chains in **both** directions from it. A good default. |
| `"first"` | anchors at the first scene. |
| `"composite"` | matches everything against the median of the first `composite_window_days` days. |
| `"2024-06-15"` | anchors at the scene nearest that date, and chains in both directions. Look at the cube first and pick a clean, high-contrast one. |

## 3. Cubes That Keep Their Clouds

Matching wants a masked cube, but you may want to keep the clouds in the output, for example for a natural-looking animation. Pass the binary mask separately: the shifts are estimated on an in-memory masked copy, exactly what the masked workflow would estimate, while the exported scenes keep every pixel.

The mask is the one the builder writes with `cloud_mask_output=` (notebook 2, chapter 2.1). It may hold more dates than the cube, but every cube date must be in it.

In [ ]:
from stac2cube import get_stac_layers

cube_with_clouds = get_stac_layers(
    mission="s2",
    polygon="../polygons/test.gpkg",
    resolution=10,
    daterange=["2024-04-01", "2024-04-30"],
    bands=["blue", "green", "red", "nir"],
    max_cc=100,
    cloud_masking=True,
    keep_clouds=True,                                  # pixels stay in
    cloud_mask_output="../results/test_cr_mask.nc",    # mask goes to its own file
    output="../results/test_clouds_kept.nc",
    q=True,
)

In [ ]:
out = coregister_cube(
    input_path="../results/test_clouds_kept.nc",
    cloud_mask="../results/test_cr_mask.nc",
    grid_size=7,
    first_scene_mode="auto",
    output_path="../results/test_clouds_kept_cr.nc",
)

## 4. Check the Result

Side-by-side animations are the quickest check: the jitter either stops or it does not.

In [ ]:
from stac2cube import save_timeseries_gif, open_cube

with open_cube("../results/test.nc") as ds:
    before = ds["Time_Series"].load()

with open_cube("../results/test_cr.nc") as ds:
    after = ds["Time_Series"].load()

save_timeseries_gif(before, "../animations/test_before.gif", display_mode="rgb", fps=3)
save_timeseries_gif(after, "../animations/test_after.gif", display_mode="rgb", fps=3)

> The co-registered cube can hold fewer dates than the original, because unmatched scenes are dropped. Compare the animations knowing that, rather than counting frames.

In [ ]:
print("before:", before.time.size, "scenes")
print("after :", after.time.size, "scenes")

## 5. Spectral Profile Before and After

Pulls the time series of a single pixel out of both cubes and plots them together. A pixel on a sharp boundary, for example a field edge or a shoreline, is where mis-registration shows most clearly: before co-registration the value jumps between the two land covers from date to date, afterwards it follows one of them.

Click a pixel on the map, use the zoom tools at the top to navigate.

In [ ]:
from stac2cube import spectral_profiler

spectral_profiler(
    "../results/test.nc",          # before
    "../results/test_cr.nc",       # after
    band="ndvi",
    rgb_time="median",             # background image: "first" or "median"
);